# Q-CaDD Workflow Notebook



## Molecular Processing


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent


import pandas as pd

input_csv_path = ROOT / "Data/Stage_1_Processsing_Extrapolation_data/EGFR_Inhibitors_SMILES_DB.csv"
output_csv_path = ROOT / "Data/Stage_1_Processsing_Extrapolation_data/EGFR_Inhibitors_SMILES_processed.csv"

source_df = pd.read_csv(input_csv_path)
processed_df = source_df.rename(columns={"SMILE": "SMILES"})[["Name", "SMILES"]].dropna(subset=["SMILES"]).drop_duplicates(subset=["SMILES"]).reset_index(drop=True)
processed_df.to_csv(output_csv_path, index=False)

print(f"Saved {len(processed_df)} processed ligands to {output_csv_path}")


In [ ]:
import random
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent


import pandas as pd
import selfies as sf
from rdkit import Chem

random.seed(42)

input_csv_path = ROOT / "Data/Stage_1_Processsing_Extrapolation_data/EGFR_Inhibitors_SMILES_processed.csv"
output_csv_path = ROOT / "Data/Stage_1_Processsing_Extrapolation_data/EGFR_Inhibitors_SMILES_Selfies.csv"
num_random_smiles = 10
mutation_depths = [1, 2, 3, 4, 5]
expected_variants_per_parent = 50

semantic_alphabet = list(sf.get_semantic_robust_alphabet())


In [ ]:
def randomize_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=False, doRandom=True)


def sanitize_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


def mutate_selfies(selfies_string: str, num_mutations: int) -> str:
    tokens = list(sf.split_selfies(selfies_string))
    max_length = len(tokens) + num_mutations

    for _ in range(num_mutations):
        operation = random.choice(["insert", "replace", "delete"])
        if operation == "insert":
            insert_at = random.randint(0, len(tokens))
            tokens.insert(insert_at, random.choice(semantic_alphabet))
        elif operation == "replace" and tokens:
            replace_at = random.randrange(len(tokens))
            tokens[replace_at] = random.choice(semantic_alphabet)
        elif operation == "delete" and len(tokens) > 1:
            delete_at = random.randrange(len(tokens))
            del tokens[delete_at]

        if len(tokens) > max_length:
            tokens = tokens[:max_length]

    return "".join(tokens)


In [ ]:
source_df = pd.read_csv(input_csv_path)
generated_records = []

for parent_index, parent_smiles in enumerate(source_df["SMILES"].dropna(), start=1):
    randomized_smiles = [randomize_smiles(parent_smiles) for _ in range(num_random_smiles)]
    randomized_smiles = [value for value in randomized_smiles if value]

    generated_smiles = set()
    for randomized in randomized_smiles:
        parent_selfies = sf.encoder(randomized)
        for depth in mutation_depths:
            mutated_selfies = mutate_selfies(parent_selfies, depth)
            decoded = sf.decoder(mutated_selfies)
            canonical = sanitize_smiles(decoded)
            if canonical:
                generated_smiles.add(canonical)

    for generated_smiles_value in sorted(generated_smiles):
        generated_records.append({
            "ParentIndex": parent_index,
            "ParentSMILES": parent_smiles,
            "GeneratedSMILES": generated_smiles_value,
        })

expanded_df = pd.DataFrame(generated_records).drop_duplicates(subset=["GeneratedSMILES"]).reset_index(drop=True)
expanded_df.to_csv(output_csv_path, index=False)

print(f"Generated {len(expanded_df)} unique ligands and saved them to {output_csv_path}")
print(f"Target scale per parent was approximately {expected_variants_per_parent} variants before deduplication")


## Property Filtering


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent


import pandas as pd
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Lipinski, QED

input_csv_path = ROOT / "Data/Stage_1_Processsing_Extrapolation_data/EGFR_Inhibitors_SMILES_Selfies.csv"
ghose_output_csv_path = ROOT / "Data/Stage_2_filters_data/EGFR_Inhibitors_SMILES_ghose.csv"
qed_output_csv_path = ROOT / "Data/Stage_2_filters_data/EGFR_Inhibitors_SMILES_ghose_QED.csv"


In [ ]:
def passes_paper_ghose(smiles: str) -> dict | None:
    if smiles is None:
        return None
    smiles = str(smiles).strip()
    if not smiles:
        return None

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    molecular_weight = Descriptors.MolWt(mol)
    log_p = Crippen.MolLogP(mol)
    rotatable_bonds = Lipinski.NumRotatableBonds(mol)
    hydrogen_bond_donors = Lipinski.NumHDonors(mol)
    hydrogen_bond_acceptors = Lipinski.NumHAcceptors(mol)

    passed = (
        160 <= molecular_weight <= 480
        and -0.4 <= log_p <= 5.6
        and 0 <= rotatable_bonds <= 10
        and 0 <= hydrogen_bond_donors <= 5
        and 1 <= hydrogen_bond_acceptors <= 10
    )

    if not passed:
        return None

    return {
        "SMILES": smiles,
        "MolWt": molecular_weight,
        "LogP": log_p,
        "RotatableBonds": rotatable_bonds,
        "HDonors": hydrogen_bond_donors,
        "HAcceptors": hydrogen_bond_acceptors,
    }


In [ ]:
source_df = pd.read_csv(input_csv_path)
smiles_column = "SMILES" if "SMILES" in source_df.columns else source_df.columns[-1]
smiles_series = source_df[smiles_column].dropna().astype(str).str.strip()
smiles_series = smiles_series[smiles_series.ne("")]

ghose_records = []
for smiles in smiles_series:
    result = passes_paper_ghose(smiles)
    if result is not None:
        ghose_records.append(result)

ghose_df = pd.DataFrame(ghose_records).drop_duplicates(subset=["SMILES"]).reset_index(drop=True)
ghose_df.to_csv(ghose_output_csv_path, index=False)
print(f"Saved {len(ghose_df)} Ghose-filtered ligands to {ghose_output_csv_path}")

qed_records = []
for smiles in ghose_df["SMILES"].dropna().astype(str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue
    qed_score = QED.qed(mol)
    if qed_score >= 0.5:
        qed_records.append({"SMILES": smiles, "QED": qed_score})

qed_df = pd.DataFrame(qed_records).drop_duplicates(subset=["SMILES"]).reset_index(drop=True)
qed_df.to_csv(qed_output_csv_path, index=False)
print(f"Saved {len(qed_df)} Ghose + QED filtered ligands to {qed_output_csv_path}")


## Docking Preparation and Extraction


In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")

input_csv_path = ROOT / "Data/Stage_2_filters_data/EGFR_Inhibitors_SMILES_ghose_QED.csv"
ligand_pdb_dir = ROOT / "Data/Stage_3_molecular_docking_data/Ligands_EGFR_PDB"
ligand_pdbqt_dir = ROOT / "Data/Stage_3_molecular_docking_data/Ligands_EGFR_PDBQT"
mgltools_python_path = Path("/opt/mgltools/bin/pythonsh")
prepare_ligand_script_path = Path("/opt/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py")

ligand_pdb_dir.mkdir(parents=True, exist_ok=True)
ligand_pdbqt_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
ligand_df = pd.read_csv(input_csv_path)

generated_count = 0
skipped_invalid = 0
skipped_embedding = 0
skipped_forcefield = 0

for ligand_index, smiles in enumerate(ligand_df["SMILES"].dropna().astype(str), start=1):
    smiles = smiles.strip()
    if not smiles:
        skipped_invalid += 1
        continue

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        skipped_invalid += 1
        continue

    mol = Chem.AddHs(mol)
    embed_status = AllChem.EmbedMolecule(mol, randomSeed=42)
    if embed_status != 0:
        skipped_embedding += 1
        continue

    try:
        if AllChem.UFFHasAllMoleculeParams(mol):
            AllChem.UFFOptimizeMolecule(mol)
        elif AllChem.MMFFHasAllMoleculeParams(mol):
            mmff_props = AllChem.MMFFGetMoleculeProperties(mol)
            if mmff_props is None:
                skipped_forcefield += 1
                continue
            AllChem.MMFFOptimizeMolecule(mol, mmffVariant="MMFF94")
        else:
            skipped_forcefield += 1
            continue
    except Exception:
        skipped_forcefield += 1
        continue

    pdb_path = ligand_pdb_dir / f"{ligand_index}_EGFR.pdb"
    Chem.MolToPDBFile(mol, str(pdb_path))
    generated_count += 1

print(f"Generated {generated_count} PDB files in {ligand_pdb_dir}")
print(f"Skipped invalid SMILES: {skipped_invalid}")
print(f"Skipped failed embeddings: {skipped_embedding}")
print(f"Skipped unsupported force-field molecules: {skipped_forcefield}")


In [ ]:
for pdb_path in sorted(ligand_pdb_dir.glob("*.pdb")):
    output_pdbqt_path = ligand_pdbqt_dir / f"{pdb_path.stem}.pdbqt"
    command = [
        str(mgltools_python_path),
        str(prepare_ligand_script_path),
        "-l",
        str(pdb_path),
        "-o",
        str(output_pdbqt_path),
        "-v",
    ]
    subprocess.run(command, check=True)

print(f"Generated PDBQT files in {ligand_pdbqt_dir}")


In [ ]:
import subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent


receptor_pdbqt_path = ROOT / "Data/Stage_3_molecular_docking_data/EGFR_Protein.pdbqt"
ligand_pdbqt_dir = ROOT / "Data/Stage_3_molecular_docking_data/Ligands_EGFR_PDBQT"
docking_output_dir = ROOT / "Data/Stage_3_molecular_docking_data/Docking_data"
vina_executable_path = Path("/opt/homebrew/bin/vina")

docking_output_dir.mkdir(parents=True, exist_ok=True)

grid_center = {"center_x": 23.00, "center_y": 0.00, "center_z": 56.00}
grid_size = {"size_x": 30.0, "size_y": 30.0, "size_z": 30.0}
exhaustiveness = 8
num_modes = 9


In [ ]:
for ligand_path in sorted(ligand_pdbqt_dir.glob("*.pdbqt")):
    output_path = docking_output_dir / f"out_{ligand_path.name}"
    command = [
        str(vina_executable_path),
        "--receptor", str(receptor_pdbqt_path),
        "--ligand", str(ligand_path),
        "--out", str(output_path),
        "--center_x", str(grid_center["center_x"]),
        "--center_y", str(grid_center["center_y"]),
        "--center_z", str(grid_center["center_z"]),
        "--size_x", str(grid_size["size_x"]),
        "--size_y", str(grid_size["size_y"]),
        "--size_z", str(grid_size["size_z"]),
        "--exhaustiveness", str(exhaustiveness),
        "--num_modes", str(num_modes),
    ]
    subprocess.run(command, check=True)

print(f"Docking finished. Results are stored in {docking_output_dir}")


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent


import pandas as pd

docking_output_dir = ROOT / "Data/Stage_3_molecular_docking_data/Docking_data"
filtered_ligands_csv_path = ROOT / "Data/Stage_2_filters_data/EGFR_Inhibitors_SMILES_ghose_QED.csv"
docking_summary_csv_path = ROOT / "Data/Stage_3_molecular_docking_data/docking_output_data.csv"


In [ ]:
def extract_best_affinity(pdbqt_path: Path) -> float | None:
    affinities = []
    with pdbqt_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.startswith("REMARK VINA RESULT"):
                affinities.append(float(line.split()[3]))
    return min(affinities) if affinities else None


In [ ]:
ligand_df = pd.read_csv(filtered_ligands_csv_path)
records = []

for ligand_index, smiles in enumerate(ligand_df["SMILES"], start=1):
    docking_file = docking_output_dir / f"out_{ligand_index}_EGFR.pdbqt"
    if not docking_file.exists():
        continue

    best_affinity = extract_best_affinity(docking_file)
    if best_affinity is None:
        continue

    records.append({
        "File": docking_file.name,
        "Affinity": best_affinity,
        "SMILES": smiles,
    })

summary_df = pd.DataFrame(records)
summary_df.to_csv(docking_summary_csv_path, index=False)
print(f"Saved docking summary for {len(summary_df)} ligands to {docking_summary_csv_path}")


## Feature Engineering and Modeling


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, MACCSkeys
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from qiskit.circuit.library import ZFeatureMap
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.kernels import FidelityStatevectorKernel


TARGET_COLUMN = "NR-AR"
RANDOM_STATE = 42
ENSEMBLE_WEIGHTS = {"qsvm": 0.01, "nn": 0.80, "svm": 0.09, "rf": 0.10}


@dataclass
class PreparedFeatureData:
    train_df: pd.DataFrame
    test_df: pd.DataFrame
    X_train_selected: pd.DataFrame
    X_test_selected: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series
    qsvm_pca: PCA
    X_train_qsvm: np.ndarray
    X_test_qsvm: np.ndarray
    selected_columns: pd.Index


class FeedForwardNN(torch.nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 512),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(512, 1),
        )

    def forward(self, inputs):
        return self.network(inputs)


def set_random_seeds(random_state: int = RANDOM_STATE) -> None:
    np.random.seed(random_state)
    torch.manual_seed(random_state)


def smiles_to_mol(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return mol


def build_ecfp4(mol, n_bits: int = 2048):
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)


def build_maccs(mol):
    return MACCSkeys.GenMACCSKeys(mol)


def bitvect_to_array(bitvect) -> np.ndarray:
    array = np.zeros((bitvect.GetNumBits(),), dtype=int)
    DataStructs.ConvertToNumpyArray(bitvect, array)
    return array


def summarize_similarity(query_fp, reference_fps) -> tuple[float, float]:
    if not reference_fps:
        return 0.0, 0.0
    similarities = DataStructs.BulkTanimotoSimilarity(query_fp, reference_fps)
    return float(np.mean(similarities)), float(np.max(similarities))


def build_feature_frame(df: pd.DataFrame, active_reference_fps, inactive_reference_fps) -> pd.DataFrame:
    feature_rows = []
    for smiles in df["SMILES"]:
        mol = smiles_to_mol(smiles)
        ecfp4 = build_ecfp4(mol)
        maccs = build_maccs(mol)

        mean_active, _ = summarize_similarity(ecfp4, active_reference_fps)
        mean_inactive, _ = summarize_similarity(ecfp4, inactive_reference_fps)

        row = np.concatenate(
            [
                bitvect_to_array(maccs),
                bitvect_to_array(ecfp4),
                np.array([mean_active, mean_inactive, mean_active - mean_inactive]),
            ]
        )
        feature_rows.append(row)

    return pd.DataFrame(feature_rows)


def load_split(train_csv_path: Path, test_csv_path: Path, target_column: str = TARGET_COLUMN) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_csv(train_csv_path)
    test_df = pd.read_csv(test_csv_path)
    for df in (train_df, test_df):
        if "SMILES" not in df.columns or target_column not in df.columns:
            raise ValueError("Input split files must contain SMILES and target columns")
    return train_df, test_df


def prepare_feature_data(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_column: str = TARGET_COLUMN,
    random_state: int = RANDOM_STATE,
) -> PreparedFeatureData:
    train_fps = [build_ecfp4(smiles_to_mol(smiles)) for smiles in train_df["SMILES"]]
    active_reference_fps = [fp for fp, label in zip(train_fps, train_df[target_column]) if label == 1]
    inactive_reference_fps = [fp for fp, label in zip(train_fps, train_df[target_column]) if label == 0]

    X_train_full = build_feature_frame(train_df, active_reference_fps, inactive_reference_fps)
    X_test_full = build_feature_frame(test_df, active_reference_fps, inactive_reference_fps)
    y_train = train_df[target_column].astype(int).reset_index(drop=True)
    y_test = test_df[target_column].astype(int).reset_index(drop=True)

    selector = RandomForestClassifier(n_estimators=1000, random_state=random_state, n_jobs=-1)
    selector.fit(X_train_full, y_train)
    selected_columns = X_train_full.columns[np.argsort(selector.feature_importances_)[::-1][:49]]

    X_train_selected = X_train_full[selected_columns].reset_index(drop=True)
    X_test_selected = X_test_full[selected_columns].reset_index(drop=True)

    qsvm_pca = PCA(n_components=1, random_state=random_state)
    X_train_qsvm = qsvm_pca.fit_transform(X_train_selected)
    X_test_qsvm = qsvm_pca.transform(X_test_selected)

    return PreparedFeatureData(
        train_df=train_df.reset_index(drop=True),
        test_df=test_df.reset_index(drop=True),
        X_train_selected=X_train_selected,
        X_test_selected=X_test_selected,
        y_train=y_train,
        y_test=y_test,
        qsvm_pca=qsvm_pca,
        X_train_qsvm=X_train_qsvm,
        X_test_qsvm=X_test_qsvm,
        selected_columns=selected_columns,
    )


def export_prepared_feature_data(
    prepared: PreparedFeatureData,
    selected_train_output_path: Path,
    selected_test_output_path: Path,
    qsvm_train_output_path: Path,
    qsvm_test_output_path: Path,
    target_column: str = TARGET_COLUMN,
) -> None:
    selected_train_df = pd.concat(
        [
            prepared.train_df[["SMILES"]].reset_index(drop=True),
            prepared.X_train_selected,
            prepared.y_train.rename(target_column),
        ],
        axis=1,
    )
    selected_test_df = pd.concat(
        [
            prepared.test_df[["SMILES"]].reset_index(drop=True),
            prepared.X_test_selected,
            prepared.y_test.rename(target_column),
        ],
        axis=1,
    )

    qsvm_train_df = pd.DataFrame(
        {"SMILES": prepared.train_df["SMILES"], "PC1": prepared.X_train_qsvm[:, 0], target_column: prepared.y_train}
    )
    qsvm_test_df = pd.DataFrame(
        {"SMILES": prepared.test_df["SMILES"], "PC1": prepared.X_test_qsvm[:, 0], target_column: prepared.y_test}
    )

    selected_train_df.to_csv(selected_train_output_path, index=False)
    selected_test_df.to_csv(selected_test_output_path, index=False)
    qsvm_train_df.to_csv(qsvm_train_output_path, index=False)
    qsvm_test_df.to_csv(qsvm_test_output_path, index=False)


def fit_qsvm(X_train_qsvm: np.ndarray, y_train: np.ndarray):
    feature_map = ZFeatureMap(feature_dimension=1, reps=1)
    quantum_kernel = FidelityStatevectorKernel(feature_map=feature_map)
    model = QSVC(quantum_kernel=quantum_kernel, probability=True)
    model.fit(X_train_qsvm, y_train)
    return model


def fit_random_forest(X_train: np.ndarray, y_train: np.ndarray, random_state: int = RANDOM_STATE):
    model = RandomForestClassifier(n_estimators=1000, random_state=random_state, n_jobs=-1)
    model.fit(X_train, y_train)
    return model


def fit_classical_svm(X_train: np.ndarray, y_train: np.ndarray, random_state: int = RANDOM_STATE):
    model = make_pipeline(StandardScaler(), SVC(kernel="sigmoid", probability=True, random_state=random_state))
    model.fit(X_train, y_train)
    return model


def fit_neural_network(
    X_train: np.ndarray,
    y_train: np.ndarray,
    random_state: int = RANDOM_STATE,
):
    set_random_seeds(random_state)
    model = FeedForwardNN(X_train.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1, weight_decay=1e-6)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)

    model.train()
    for _ in range(10):
        optimizer.zero_grad()
        logits = model(X_train_tensor)
        loss = loss_fn(logits, y_train_tensor)
        loss.backward()
        optimizer.step()
    return model


def predict_neural_network(model: FeedForwardNN, X: np.ndarray) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32)
        return torch.sigmoid(model(X_tensor)).numpy().ravel()


def train_models(
    prepared: PreparedFeatureData,
    random_state: int = RANDOM_STATE,
):
    y_train = prepared.y_train.to_numpy(dtype=int)
    X_train = prepared.X_train_selected.to_numpy(dtype=float)

    qsvm = fit_qsvm(prepared.X_train_qsvm, y_train)
    rf = fit_random_forest(X_train, y_train, random_state=random_state)
    svm = fit_classical_svm(X_train, y_train, random_state=random_state)
    nn = fit_neural_network(X_train, y_train, random_state=random_state)

    return {"qsvm": qsvm, "rf": rf, "svm": svm, "nn": nn}


def predict_model_probabilities(models: dict, prepared: PreparedFeatureData) -> dict[str, np.ndarray]:
    X_test = prepared.X_test_selected.to_numpy(dtype=float)
    probabilities = {
        "qsvm": models["qsvm"].predict_proba(prepared.X_test_qsvm)[:, 1],
        "rf": models["rf"].predict_proba(X_test)[:, 1],
        "svm": models["svm"].predict_proba(X_test)[:, 1],
        "nn": predict_neural_network(models["nn"], X_test),
    }
    probabilities["ensemble"] = ensemble_probabilities(probabilities)
    return probabilities


def predict_ligand_probabilities(
    models: dict,
    prepared: PreparedFeatureData,
    ligands_df: pd.DataFrame,
) -> dict[str, np.ndarray]:
    train_fps = [build_ecfp4(smiles_to_mol(smiles)) for smiles in prepared.train_df["SMILES"]]
    active_reference_fps = [fp for fp, label in zip(train_fps, prepared.y_train) if label == 1]
    inactive_reference_fps = [fp for fp, label in zip(train_fps, prepared.y_train) if label == 0]

    X_ligands_full = build_feature_frame(ligands_df, active_reference_fps, inactive_reference_fps)
    X_ligands_selected = X_ligands_full[prepared.selected_columns].to_numpy(dtype=float)
    X_ligands_qsvm = prepared.qsvm_pca.transform(X_ligands_selected)

    probabilities = {
        "qsvm": models["qsvm"].predict_proba(X_ligands_qsvm)[:, 1],
        "rf": models["rf"].predict_proba(X_ligands_selected)[:, 1],
        "svm": models["svm"].predict_proba(X_ligands_selected)[:, 1],
        "nn": predict_neural_network(models["nn"], X_ligands_selected),
    }
    probabilities["ensemble"] = ensemble_probabilities(probabilities)
    return probabilities


def ensemble_probabilities(probabilities: dict[str, np.ndarray], weights: dict[str, float] = ENSEMBLE_WEIGHTS) -> np.ndarray:
    return (
        weights["qsvm"] * probabilities["qsvm"]
        + weights["nn"] * probabilities["nn"]
        + weights["svm"] * probabilities["svm"]
        + weights["rf"] * probabilities["rf"]
    )


def evaluate_probabilities(y_true: np.ndarray, probabilities: dict[str, np.ndarray]) -> pd.DataFrame:
    label_map = {
        "qsvm": "QSVM",
        "nn": "Neural Network",
        "rf": "Random Forest",
        "svm": "Support Vector Machine",
        "ensemble": "Ensemble",
    }
    rows = []
    for key in ["qsvm", "nn", "rf", "svm", "ensemble"]:
        preds = (probabilities[key] >= 0.5).astype(int)
        rows.append(
            {
                "Model": label_map[key],
                "AUC_ROC": roc_auc_score(y_true, probabilities[key]),
                "Accuracy": accuracy_score(y_true, preds),
                "ConfusionMatrix": confusion_matrix(y_true, preds).tolist(),
            }
        )
    return pd.DataFrame(rows).sort_values("AUC_ROC", ascending=False).reset_index(drop=True)


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

import pandas as pd

train_csv_path = ROOT / "Data/Stage_4_QSVM_data/NR-AR_train_no_SMOTEEN.csv"
test_csv_path = ROOT / "Data/Stage_4_QSVM_data/NR-AR_test.csv"
selected_train_output_path = ROOT / "Data/Stage_4_QSVM_data/selected_train_features.csv"
selected_test_output_path = ROOT / "Data/Stage_4_QSVM_data/selected_test_features.csv"
qsvm_train_output_path = ROOT / "Data/Stage_4_QSVM_data/qsvm_train_pca.csv"
qsvm_test_output_path = ROOT / "Data/Stage_4_QSVM_data/qsvm_test_pca.csv"


In [ ]:
train_df = pd.read_csv(train_csv_path).rename(columns={"nr-ar": TARGET_COLUMN})[["SMILES", TARGET_COLUMN]]
test_df = pd.read_csv(test_csv_path)[["SMILES", TARGET_COLUMN]]
prepared = prepare_feature_data(
    train_df=train_df,
    test_df=test_df,
    target_column=TARGET_COLUMN,
    random_state=RANDOM_STATE,
)

export_prepared_feature_data(
    prepared=prepared,
    selected_train_output_path=selected_train_output_path,
    selected_test_output_path=selected_test_output_path,
    qsvm_train_output_path=qsvm_train_output_path,
    qsvm_test_output_path=qsvm_test_output_path,
    target_column=TARGET_COLUMN,
)

print("Selected feature matrices and QSVM-ready PCA projections were saved successfully")


In [ ]:
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

import matplotlib.pyplot as plt
import pandas as pd
from imblearn.combine import SMOTEENN


In [ ]:
train_csv_path = ROOT / "Data/Stage_4_QSVM_data/NR-AR_train_no_SMOTEEN.csv"
test_csv_path = ROOT / "Data/Stage_4_QSVM_data/NR-AR_test.csv"
resampled_train_output_path = ROOT / "Data/Stage_4_QSVM_data/resampled_train.csv"

set_random_seeds(RANDOM_STATE)


In [ ]:
train_df = pd.read_csv(train_csv_path).rename(columns={"nr-ar": TARGET_COLUMN})
test_df = pd.read_csv(test_csv_path)

prepared = prepare_feature_data(
    train_df=train_df[["SMILES", TARGET_COLUMN]].copy(),
    test_df=test_df[["SMILES", TARGET_COLUMN]].copy(),
    target_column=TARGET_COLUMN,
    random_state=RANDOM_STATE,
)

X_train = prepared.X_train_selected.copy()
y_train = prepared.y_train.copy()
X_test = prepared.X_test_selected.copy()
y_test = prepared.y_test.copy()

print("Shape of train data for the assay", X_train.shape)
print("Shape of train labels for the assay", y_train.shape)
print("Shape of test data for the assay", X_test.shape)
print("Shape of test labels for the assay", y_test.shape)


In [ ]:
counter_train = Counter(y_train)
counter_test = Counter(y_test)
print("Imbalanced ratio for the assay {:0.2f}".format(counter_train[0] / counter_train[1]))
print("Imbalanced ratio for the assay {:0.2f}".format(counter_test[0] / counter_test[1]))

pd.Series(counter_train).sort_index().plot(kind="bar", title="NR-AR Train Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


In [ ]:
sampler = SMOTEENN(random_state=RANDOM_STATE)
X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)

print("Transforming SMOTEENN")
print("Resampled shape", X_resampled.shape)
print("Resampled class counts")
print(pd.Series(y_resampled).value_counts().sort_index())


In [ ]:
resampled_output_df = pd.concat(
    [pd.DataFrame(X_resampled, columns=X_train.columns), pd.Series(y_resampled, name=TARGET_COLUMN)],
    axis=1,
)
resampled_output_df.to_csv(resampled_train_output_path, index=False)
print(f"Saved SMOTE-ENN training matrix to {resampled_train_output_path}")
resampled_output_df.head()


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

import pandas as pd

train_csv_path = ROOT / "Data/Stage_4_QSVM_data/NR-AR_train_no_SMOTEEN.csv"
test_csv_path = ROOT / "Data/Stage_4_QSVM_data/NR-AR_test.csv"


In [ ]:
set_random_seeds(RANDOM_STATE)
train_df = pd.read_csv(train_csv_path).rename(columns={"nr-ar": TARGET_COLUMN})[["SMILES", TARGET_COLUMN]]
test_df = pd.read_csv(test_csv_path)[["SMILES", TARGET_COLUMN]]
prepared = prepare_feature_data(
    train_df=train_df,
    test_df=test_df,
    target_column=TARGET_COLUMN,
    random_state=RANDOM_STATE,
)
models = train_models(prepared, random_state=RANDOM_STATE)
probabilities = predict_model_probabilities(models, prepared)


In [ ]:
results_df = evaluate_probabilities(prepared.y_test.to_numpy(dtype=int), probabilities)
results_df


In [ ]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn.combine import SMOTEENN
from rdkit import RDLogger
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

TRAIN_CSV = ROOT / "Data/Stage_4_QSVM_data/NR-AR_train_no_SMOTEEN.csv"
TEST_CSV = ROOT / "Data/Stage_4_QSVM_data/NR-AR_test.csv"
VALIDATION_SIZE = 0.2
OUTPUT_CSV = ROOT / "Data/Stage_4_QSVM_data/stage4_final_results.csv"


def load_raw_splits(train_csv: Path, test_csv: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_csv(train_csv).rename(columns={"nr-ar": TARGET_COLUMN})
    test_df = pd.read_csv(test_csv)
    required = {"SMILES", TARGET_COLUMN}
    if not required.issubset(train_df.columns):
        raise ValueError(f"Train CSV must contain {sorted(required)}")
    if not required.issubset(test_df.columns):
        raise ValueError(f"Test CSV must contain {sorted(required)}")
    return train_df[["SMILES", TARGET_COLUMN]].copy(), test_df[["SMILES", TARGET_COLUMN]].copy()


def match_test_imbalance_split(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    validation_size: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    y = train_df[TARGET_COLUMN].astype(int).to_numpy()
    test_pos_rate = float(test_df[TARGET_COLUMN].astype(int).mean())
    n_val = max(1, int(round(len(train_df) * validation_size)))

    pos_idx = np.flatnonzero(y == 1)
    neg_idx = np.flatnonzero(y == 0)
    target_pos = int(round(n_val * test_pos_rate))
    target_pos = max(1, min(target_pos, len(pos_idx)))
    target_neg = n_val - target_pos
    if target_neg > len(neg_idx):
        target_neg = len(neg_idx)
        target_pos = min(n_val - target_neg, len(pos_idx))

    rng = np.random.default_rng(RANDOM_STATE)
    val_pos = rng.choice(pos_idx, size=target_pos, replace=False)
    val_neg = rng.choice(neg_idx, size=target_neg, replace=False)
    val_idx = np.concatenate([val_pos, val_neg])
    rng.shuffle(val_idx)

    mask = np.ones(len(train_df), dtype=bool)
    mask[val_idx] = False
    fit_df = train_df.iloc[np.flatnonzero(mask)].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)
    return fit_df, val_df


def build_descriptor_frames(
    fit_df: pd.DataFrame,
    query_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    fit_fps = [build_ecfp4(smiles_to_mol(smiles)) for smiles in fit_df["SMILES"]]
    active_reference_fps = [fp for fp, label in zip(fit_fps, fit_df[TARGET_COLUMN].astype(int)) if label == 1]
    inactive_reference_fps = [fp for fp, label in zip(fit_fps, fit_df[TARGET_COLUMN].astype(int)) if label == 0]

    X_fit = build_feature_frame(fit_df, active_reference_fps, inactive_reference_fps)
    X_query = build_feature_frame(query_df, active_reference_fps, inactive_reference_fps)
    X_fit.columns = X_fit.columns.map(str)
    X_query.columns = X_query.columns.map(str)
    return X_fit, X_query


def evaluate_named_probabilities(name: str, y_true: np.ndarray, probs: np.ndarray) -> dict[str, object]:
    preds = (probs >= 0.5).astype(int)
    return {
        "Model": name,
        "AUC_ROC": roc_auc_score(y_true, probs),
        "Accuracy": accuracy_score(y_true, preds),
        "ConfusionMatrix": confusion_matrix(y_true, preds).tolist(),
    }


def ensemble_stage4_probabilities(probabilities: dict[str, np.ndarray]) -> np.ndarray:
    return (
        ENSEMBLE_WEIGHTS["qsvm"] * probabilities["qsvm"]
        + ENSEMBLE_WEIGHTS["nn"] * probabilities["nn"]
        + ENSEMBLE_WEIGHTS["svm"] * probabilities["svm"]
        + ENSEMBLE_WEIGHTS["rf"] * probabilities["rf"]
    )


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.combine import SMOTEENN
from rdkit import RDLogger
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, roc_auc_score, roc_curve

TRAIN_CSV = ROOT / "Data/Stage_4_QSVM_data/NR-AR_train_no_SMOTEEN.csv"
TEST_CSV = ROOT / "Data/Stage_4_QSVM_data/NR-AR_test.csv"
VALIDATION_SIZE = 0.2
OUTPUT_CSV = ROOT / "Data/Stage_4_QSVM_data/stage4_final_results.csv"

RDLogger.DisableLog("rdApp.*")
set_random_seeds(RANDOM_STATE)


In [ ]:
def load_raw_splits(train_csv: Path, test_csv: Path):
    train_df = pd.read_csv(train_csv).rename(columns={"nr-ar": TARGET_COLUMN})
    test_df = pd.read_csv(test_csv)
    required = {"SMILES", TARGET_COLUMN}
    if not required.issubset(train_df.columns):
        raise ValueError(f"Train CSV must contain {sorted(required)}")
    if not required.issubset(test_df.columns):
        raise ValueError(f"Test CSV must contain {sorted(required)}")
    return train_df[["SMILES", TARGET_COLUMN]].copy(), test_df[["SMILES", TARGET_COLUMN]].copy()


def match_test_imbalance_split(train_df: pd.DataFrame, test_df: pd.DataFrame, validation_size: float):
    y = train_df[TARGET_COLUMN].astype(int).to_numpy()
    test_pos_rate = float(test_df[TARGET_COLUMN].astype(int).mean())
    n_val = max(1, int(round(len(train_df) * validation_size)))

    pos_idx = np.flatnonzero(y == 1)
    neg_idx = np.flatnonzero(y == 0)
    target_pos = int(round(n_val * test_pos_rate))
    target_pos = max(1, min(target_pos, len(pos_idx)))
    target_neg = n_val - target_pos
    if target_neg > len(neg_idx):
        target_neg = len(neg_idx)
        target_pos = min(n_val - target_neg, len(pos_idx))

    rng = np.random.default_rng(RANDOM_STATE)
    val_pos = rng.choice(pos_idx, size=target_pos, replace=False)
    val_neg = rng.choice(neg_idx, size=target_neg, replace=False)
    val_idx = np.concatenate([val_pos, val_neg])
    rng.shuffle(val_idx)

    mask = np.ones(len(train_df), dtype=bool)
    mask[val_idx] = False
    fit_df = train_df.iloc[np.flatnonzero(mask)].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)
    return fit_df, val_df


def build_descriptor_frames(fit_df: pd.DataFrame, query_df: pd.DataFrame):
    fit_fps = [build_ecfp4(smiles_to_mol(smiles)) for smiles in fit_df["SMILES"]]
    active_reference_fps = [fp for fp, label in zip(fit_fps, fit_df[TARGET_COLUMN].astype(int)) if label == 1]
    inactive_reference_fps = [fp for fp, label in zip(fit_fps, fit_df[TARGET_COLUMN].astype(int)) if label == 0]

    X_fit = build_feature_frame(fit_df, active_reference_fps, inactive_reference_fps)
    X_query = build_feature_frame(query_df, active_reference_fps, inactive_reference_fps)
    X_fit.columns = X_fit.columns.map(str)
    X_query.columns = X_query.columns.map(str)
    return X_fit, X_query


def evaluate_model(name: str, y_true: np.ndarray, probs: np.ndarray):
    preds = (probs >= 0.5).astype(int)
    return {
        "Model": name,
        "AUC_ROC": roc_auc_score(y_true, probs),
        "Accuracy": accuracy_score(y_true, preds),
        "ConfusionMatrix": confusion_matrix(y_true, preds).tolist(),
    }


def ensemble_probabilities(probabilities: dict[str, np.ndarray]) -> np.ndarray:
    return (
        ENSEMBLE_WEIGHTS["qsvm"] * probabilities["qsvm"]
        + ENSEMBLE_WEIGHTS["nn"] * probabilities["nn"]
        + ENSEMBLE_WEIGHTS["svm"] * probabilities["svm"]
        + ENSEMBLE_WEIGHTS["rf"] * probabilities["rf"]
    )

In [ ]:
train_df, test_df = load_raw_splits(TRAIN_CSV, TEST_CSV)
fit_df, val_df = match_test_imbalance_split(train_df, test_df, VALIDATION_SIZE)

X_fit_full, X_val_full = build_descriptor_frames(fit_df, val_df)
_, X_test_full = build_descriptor_frames(fit_df, test_df)
y_fit = fit_df[TARGET_COLUMN].astype(int).to_numpy()
y_val = val_df[TARGET_COLUMN].astype(int).to_numpy()
y_test = test_df[TARGET_COLUMN].astype(int).to_numpy()

selector = RandomForestClassifier(n_estimators=1000, random_state=RANDOM_STATE, n_jobs=-1)
selector.fit(X_fit_full, y_fit)
selected_columns = X_fit_full.columns[np.argsort(selector.feature_importances_)[::-1][:49]]

X_fit_selected = X_fit_full[selected_columns].to_numpy(dtype=float)
X_val_selected = X_val_full[selected_columns].to_numpy(dtype=float)
X_test_selected = X_test_full[selected_columns].to_numpy(dtype=float)

sampler = SMOTEENN(random_state=RANDOM_STATE)
X_fit_resampled, y_fit_resampled = sampler.fit_resample(X_fit_selected, y_fit)

pca = PCA(n_components=1, random_state=RANDOM_STATE)
X_fit_qsvm = pca.fit_transform(X_fit_resampled)
X_val_qsvm = pca.transform(X_val_selected)
X_test_qsvm = pca.transform(X_test_selected)
qsvm = fit_qsvm(X_fit_qsvm, y_fit_resampled)
rf = fit_random_forest(X_fit_resampled, y_fit_resampled, random_state=RANDOM_STATE)
svm = fit_classical_svm(X_fit_resampled, y_fit_resampled, random_state=RANDOM_STATE)
nn = fit_neural_network(X_fit_resampled, y_fit_resampled, random_state=RANDOM_STATE)

val_probs = {
    "qsvm": qsvm.predict_proba(X_val_qsvm)[:, 1],
    "rf": rf.predict_proba(X_val_selected)[:, 1],
    "svm": svm.predict_proba(X_val_selected)[:, 1],
    "nn": predict_neural_network(nn, X_val_selected),
}
val_probs["ensemble"] = ensemble_probabilities(val_probs)

test_probs = {
    "qsvm": qsvm.predict_proba(X_test_qsvm)[:, 1],
    "rf": rf.predict_proba(X_test_selected)[:, 1],
    "svm": svm.predict_proba(X_test_selected)[:, 1],
    "nn": predict_neural_network(nn, X_test_selected),
}
test_probs["ensemble"] = ensemble_probabilities(test_probs)

rows = [
    {"Split": "validation", **evaluate_model("QSVM", y_val, val_probs["qsvm"])},
    {"Split": "validation", **evaluate_model("Neural Network", y_val, val_probs["nn"])},
    {"Split": "validation", **evaluate_model("Random Forest", y_val, val_probs["rf"])},
    {"Split": "validation", **evaluate_model("Support Vector Machine", y_val, val_probs["svm"])},
    {"Split": "validation", **evaluate_model("Ensemble", y_val, val_probs["ensemble"])},
    {"Split": "test", **evaluate_model("QSVM", y_test, test_probs["qsvm"])},
    {"Split": "test", **evaluate_model("Neural Network", y_test, test_probs["nn"])},
    {"Split": "test", **evaluate_model("Random Forest", y_test, test_probs["rf"])},
    {"Split": "test", **evaluate_model("Support Vector Machine", y_test, test_probs["svm"])},
    {"Split": "test", **evaluate_model("Ensemble", y_test, test_probs["ensemble"])},
]
results_df = pd.DataFrame(rows)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_CSV, index=False)

print(f"Ensemble weights used: {ENSEMBLE_WEIGHTS}")
print(f"Validation positive rate: {float(y_val.mean()):.6f}")
print(f"Test positive rate: {float(y_test.mean()):.6f}")
print(f"Fit rows before resampling: {len(y_fit)}")
print(f"Fit rows after SMOTE-ENN: {len(y_fit_resampled)}")
print(f"QSVM training rows used: {len(y_fit_resampled)}")
results_df

In [ ]:
test_results_df = results_df[results_df["Split"] == "test"].copy().reset_index(drop=True)
test_results_df

In [ ]:
label_map = {
    "rf": "Random Forest",
    "svm": "Support Vector Machine",
    "nn": "Neural Network",
    "ensemble": "Ensemble Method",
}

display_order = ["svm", "nn", "rf", "ensemble"]
fig, axes = plt.subplots(2, 4, figsize=(20, 11), constrained_layout=False)
plt.subplots_adjust(left=0.05, right=0.98, bottom=0.06, top=0.90, wspace=0.45, hspace=0.42)

header_positions = {
    "svm": (0.25, 0.972),
    "nn": (0.75, 0.972),
    "rf": (0.25, 0.50),
    "ensemble": (0.75, 0.50),
}

for key, (x, y) in header_positions.items():
    fig.text(x, y, label_map[key], ha="center", va="top", fontsize=20, fontweight="bold")

for idx, key in enumerate(display_order):
    row = idx // 2
    col_group = idx % 2
    ax_cm = axes[row, col_group * 2]
    ax_roc = axes[row, col_group * 2 + 1]

    probs = test_probs[key]
    preds = (probs >= 0.5).astype(int)
    cm = confusion_matrix(y_test, preds)

    im = ax_cm.imshow(cm, cmap="Blues")
    fig.colorbar(im, ax=ax_cm, fraction=0.046, pad=0.03)
    ax_cm.set_title("Confusion Matrix", fontsize=11, pad=6)
    ax_cm.set_xlabel("Predicted")
    ax_cm.set_ylabel("Actual")
    ax_cm.set_xticks([0, 1], labels=["Class 0", "Class 1"])
    ax_cm.set_yticks([0, 1], labels=["Class 0", "Class 1"])

    threshold = cm.max() / 2 if cm.max() else 0
    for r in range(cm.shape[0]):
        for c in range(cm.shape[1]):
            ax_cm.text(
                c,
                r,
                f"{cm[r, c]}",
                ha="center",
                va="center",
                color="white" if cm[r, c] > threshold else "#333333",
                fontsize=10,
            )

    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_value = roc_auc_score(y_test, probs)
    ax_roc.plot(fpr, tpr, color="darkorange", linewidth=2, label=f"ROC curve (AUC = {auc_value:.2f})")
    ax_roc.plot([0, 1], [0, 1], linestyle="--", color="navy", linewidth=1.5)
    ax_roc.set_title("Receiver Operating Characteristic (ROC) Curve for NR-AR", fontsize=11, pad=6)
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_xlim(0, 1)
    ax_roc.set_ylim(0, 1.05)
    ax_roc.legend(loc="lower right")

plt.show()


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

import pandas as pd

train_csv_path = ROOT / "Data/Stage_4_QSVM_data/NR-AR_train_no_SMOTEEN.csv"
test_csv_path = ROOT / "Data/Stage_4_QSVM_data/NR-AR_test.csv"
docked_ligands_csv_path = ROOT / "Data/Stage_4_QSVM_data/binding_ligands_output.csv"
ligand_prediction_output_path = ROOT / "Data/Stage_5_datapoint_calculation_data/final_toxicity_affinity_distance.csv"


In [ ]:
set_random_seeds(RANDOM_STATE)
train_df = pd.read_csv(train_csv_path).rename(columns={"nr-ar": TARGET_COLUMN})[["SMILES", TARGET_COLUMN]]
test_df = pd.read_csv(test_csv_path)[["SMILES", TARGET_COLUMN]]
prepared = prepare_feature_data(
    train_df=train_df,
    test_df=test_df,
    target_column=TARGET_COLUMN,
    random_state=RANDOM_STATE,
)
models = train_models(prepared, random_state=RANDOM_STATE)
ligands_df = pd.read_csv(docked_ligands_csv_path)
probabilities = predict_ligand_probabilities(models, prepared, ligands_df)


In [ ]:
output_df = ligands_df.copy()
output_df["QSVM_Probability"] = probabilities["qsvm"]
output_df["NN_Probability"] = probabilities["nn"]
output_df["SVM_Probability"] = probabilities["svm"]
output_df["RF_Probability"] = probabilities["rf"]
output_df["Prediction"] = probabilities["ensemble"]
output_df["EnsembleWeight_QSVM"] = ENSEMBLE_WEIGHTS["qsvm"]
output_df["EnsembleWeight_NN"] = ENSEMBLE_WEIGHTS["nn"]
output_df["EnsembleWeight_SVM"] = ENSEMBLE_WEIGHTS["svm"]
output_df["EnsembleWeight_RF"] = ENSEMBLE_WEIGHTS["rf"]
output_df.to_csv(ligand_prediction_output_path, index=False)

print(f"Saved ligand toxicity predictions to {ligand_prediction_output_path}")


## Ranking


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent


import numpy as np
import pandas as pd

input_csv_path = ROOT / "Data/Stage_5_datapoint_calculation_data/final_toxicity_affinity_distance.csv"
output_csv_path = ROOT / "Data/Stage_5_datapoint_calculation_data/final_toxicity_affinity_distance.csv"
ideal_affinity = -12.0
ideal_toxicity = 0.0


In [ ]:
df = pd.read_csv(input_csv_path)

distance = np.sqrt((ideal_affinity - df["Affinity"].to_numpy(dtype=float)) ** 2 + (ideal_toxicity - df["Prediction"].to_numpy(dtype=float)) ** 2)
ranked_df = df.copy()
ranked_df["Distance"] = distance
ranked_df = ranked_df.sort_values("Distance", ascending=True).reset_index(drop=True)
ranked_df.to_csv(output_csv_path, index=False)

print(f"Saved ranked ligand table to {output_csv_path}")
ranked_df.head()


In [ ]:
from pathlib import Path
import subprocess

import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

erlotinib_sdf_path = ROOT / "Data/erlotinib.sdf"
if not erlotinib_sdf_path.exists():
    erlotinib_sdf_path = ROOT / "erlotinib.sdf"
if not erlotinib_sdf_path.exists():
    raise FileNotFoundError("Could not find erlotinib.sdf in Data/ or the project root")

single_ligand_dir = ROOT / "Data/Stage_3_molecular_docking_data/single_ligand_eval"
single_ligand_dir.mkdir(parents=True, exist_ok=True)

single_ligand_pdb_path = single_ligand_dir / "erlotinib.pdb"
single_ligand_pdbqt_path = single_ligand_dir / "erlotinib.pdbqt"
single_ligand_docked_path = single_ligand_dir / "out_erlotinib.pdbqt"

mgltools_python_path = Path("/opt/mgltools/bin/pythonsh")
prepare_ligand_script_path = Path("/opt/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py")
vina_executable_path = Path("/opt/homebrew/bin/vina")
receptor_pdbqt_path = ROOT / "Data/Stage_3_molecular_docking_data/EGFR_Protein.pdbqt"

supplier = Chem.SDMolSupplier(str(erlotinib_sdf_path), removeHs=False)
erlotinib_mol = next((mol for mol in supplier if mol is not None), None)
if erlotinib_mol is None:
    raise ValueError(f"No valid molecule found in {erlotinib_sdf_path}")

erlotinib_smiles = Chem.MolToSmiles(Chem.RemoveHs(erlotinib_mol))
prep_mol = Chem.AddHs(Chem.MolFromSmiles(erlotinib_smiles))
AllChem.EmbedMolecule(prep_mol, randomSeed=RANDOM_STATE)
AllChem.UFFOptimizeMolecule(prep_mol)
Chem.MolToPDBFile(prep_mol, str(single_ligand_pdb_path))

subprocess.run(
    [
        str(mgltools_python_path),
        str(prepare_ligand_script_path),
        "-l",
        str(single_ligand_pdb_path),
        "-o",
        str(single_ligand_pdbqt_path),
        "-A",
        "hydrogens",
    ],
    check=True,
)

subprocess.run(
    [
        str(vina_executable_path),
        "--receptor",
        str(receptor_pdbqt_path),
        "--ligand",
        str(single_ligand_pdbqt_path),
        "--out",
        str(single_ligand_docked_path),
        "--center_x",
        "23.0",
        "--center_y",
        "0.0",
        "--center_z",
        "56.0",
        "--size_x",
        "30.0",
        "--size_y",
        "30.0",
        "--size_z",
        "30.0",
        "--exhaustiveness",
        "8",
        "--num_modes",
        "9",
    ],
    check=True,
)

if "prepared" not in globals() or "models" not in globals():
    train_df = pd.read_csv(ROOT / "Data/Stage_4_QSVM_data/NR-AR_train_no_SMOTEEN.csv").rename(columns={"nr-ar": TARGET_COLUMN})[["SMILES", TARGET_COLUMN]]
    test_df = pd.read_csv(ROOT / "Data/Stage_4_QSVM_data/NR-AR_test.csv")[["SMILES", TARGET_COLUMN]]
    prepared = prepare_feature_data(train_df=train_df, test_df=test_df, target_column=TARGET_COLUMN, random_state=RANDOM_STATE)
    models = train_models(prepared, random_state=RANDOM_STATE)

erlotinib_affinity = extract_best_affinity(single_ligand_docked_path)
erlotinib_df = pd.DataFrame([{"Ligand": "erlotinib", "SMILES": erlotinib_smiles, "Affinity": erlotinib_affinity}])
erlotinib_probs = predict_ligand_probabilities(models, prepared, erlotinib_df[["SMILES"]])

erlotinib_result = erlotinib_df.copy()
erlotinib_result["QSVM_Probability"] = erlotinib_probs["qsvm"]
erlotinib_result["NN_Probability"] = erlotinib_probs["nn"]
erlotinib_result["SVM_Probability"] = erlotinib_probs["svm"]
erlotinib_result["RF_Probability"] = erlotinib_probs["rf"]
erlotinib_result["Ensemble_Toxicity_Prediction"] = erlotinib_probs["ensemble"]

if "ideal_affinity" in globals() and "ideal_toxicity" in globals():
    erlotinib_result["Distance"] = ((ideal_affinity - erlotinib_result["Affinity"]) ** 2 + (ideal_toxicity - erlotinib_result["Ensemble_Toxicity_Prediction"]) ** 2) ** 0.5

print(f"Docked pose saved to: {single_ligand_docked_path}")
erlotinib_result
